# Constrained Adaptive PGD

In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

### load data

I suggest these resources to get a better understanding of the data:

[1] [Medium - The Elliptic Data Set](https://medium.com/elliptic/the-elliptic-data-set-opening-up-machine-learning-on-the-blockchain-e0a343d99a14)  
[2] [Kaggle - EDA](https://www.kaggle.com/code/smlopezza/elliptic-data-set-eda-graphs-random-forest/notebook)

You can also download [here](https://www.kaggle.com/datasets/ellipticco/elliptic-data-set/code?datasetId=284040&sortBy=voteCount) the dataset.

In [4]:
# CODE FROM [2]

df_classes = pd.read_csv("../data/elliptic_bitcoin_dataset/elliptic_txs_classes.csv")
df_edges = pd.read_csv("../data/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv")
df_features = pd.read_csv("../data/elliptic_bitcoin_dataset/elliptic_txs_features.csv", header=None)

colNames1 = {'0': 'txId', 1: "Time step"}
colNames2 = {str(ii+2): "Local_feature_" + str(ii+1) for ii in range(93)}
colNames3 = {str(ii+95): "Aggregate_feature_" + str(ii+1) for ii in range(72)}
colNames = dict(colNames1, **colNames2, **colNames3)
colNames = {int(jj): item_kk for jj, item_kk in colNames.items()}
df_features = df_features.rename(columns=colNames)

df_classes.loc[df_classes['class'] == 'unknown', 'class'] = 3
df_class_feature = pd.merge(df_classes, df_features)

selected_ids = df_class_feature.loc[(df_class_feature['class'] != 3), 'txId']
df_edges_selected = df_edges.loc[df_edges['txId1'].isin(selected_ids)]
df_classes_selected = df_classes.loc[df_classes['txId'].isin(selected_ids)]
df_features_selected = df_features.loc[df_features['txId'].isin(selected_ids)]

df_class_feature_selected = pd.merge(df_classes_selected, df_features_selected)

X_train = df_class_feature_selected.loc[
    df_class_feature_selected['Time step'].between(1, 34)
].drop(columns=['txId', 'class', 'Time step'])

y_train = df_class_feature_selected.loc[
    df_class_feature_selected['Time step'].between(1, 34),
    'class'
].apply(lambda x: 0 if x == '2' else 1)


X_test = df_class_feature_selected.loc[
    df_class_feature_selected['Time step'].between(35, 49)
].drop(columns=['txId', 'class', 'Time step'])

y_test = df_class_feature_selected.loc[
    df_class_feature_selected['Time step'].between(35, 49),
    'class'
].apply(lambda x: 0 if x == '2' else 1)

### Preprocessing

In [6]:
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

N, D = X_train_t.shape
print(f"Train: {X_train_t.shape} | Test: {X_test_t.shape} | Features: {D}")

if torch.cuda.is_available():
    device = 'cuda' 
else: 
    device = 'cpu'

Train: torch.Size([29894, 165]) | Test: torch.Size([16670, 165]) | Features: 165


### Multi-Layer Perceptron (MLP)

In [7]:
class TabularMLP(nn.Module):
    def __init__(self, n_features, hidden_dims=[256, 128, 64], n_classes=2, dropout=0.3):
        super().__init__()
        layers = []
        prev_dim = n_features
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = TabularMLP(n_features=D).to(device)

# Pesi di classe per lo sbilanciamento (2% illicit)
class_counts = np.bincount(y_train.values)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = (class_weights / class_weights.sum() * 2).to(device)
print(f"Class counts: {class_counts} | Weights: {class_weights.cpu().numpy()}")

Class counts: [26432  3462] | Weights: [0.23161839 1.7683816 ]


### Training loop

In [8]:
batch_size = 1024
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss(weight=class_weights)
n_epochs = 20

for epoch in range(n_epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    model.eval()
    with torch.no_grad():
        val_preds = model(X_test_t.to(device)).argmax(dim=1).cpu().numpy()
        f1_illicit = f1_score(y_test_t.numpy(), val_preds, pos_label=1)
    print(f"Epoch {epoch+1:02d}/{n_epochs} | Loss: {total_loss/len(train_ds):.4f} | "
          f"Test F1 (illicit): {f1_illicit:.4f}")

model.eval()
with torch.no_grad():
    test_preds = model(X_test_t.to(device)).argmax(dim=1).cpu().numpy()
print("\nClassification Report (clean)")
print(classification_report(y_test_t.numpy(), test_preds, target_names=['licit', 'illicit']))

Epoch 01/20 | Loss: 0.4175 | Test F1 (illicit): 0.2512
Epoch 02/20 | Loss: 0.2420 | Test F1 (illicit): 0.3378
Epoch 03/20 | Loss: 0.1943 | Test F1 (illicit): 0.4240
Epoch 04/20 | Loss: 0.1706 | Test F1 (illicit): 0.4873
Epoch 05/20 | Loss: 0.1537 | Test F1 (illicit): 0.4875
Epoch 06/20 | Loss: 0.1434 | Test F1 (illicit): 0.5520
Epoch 07/20 | Loss: 0.1322 | Test F1 (illicit): 0.5654
Epoch 08/20 | Loss: 0.1274 | Test F1 (illicit): 0.5109
Epoch 09/20 | Loss: 0.1199 | Test F1 (illicit): 0.6074
Epoch 10/20 | Loss: 0.1133 | Test F1 (illicit): 0.5748
Epoch 11/20 | Loss: 0.1119 | Test F1 (illicit): 0.6096
Epoch 12/20 | Loss: 0.1074 | Test F1 (illicit): 0.5523
Epoch 13/20 | Loss: 0.1029 | Test F1 (illicit): 0.6288
Epoch 14/20 | Loss: 0.0973 | Test F1 (illicit): 0.6334
Epoch 15/20 | Loss: 0.0916 | Test F1 (illicit): 0.6206
Epoch 16/20 | Loss: 0.0959 | Test F1 (illicit): 0.6494
Epoch 17/20 | Loss: 0.0899 | Test F1 (illicit): 0.6292
Epoch 18/20 | Loss: 0.0944 | Test F1 (illicit): 0.6450
Epoch 19/2

## CAPGD (Constrained Adaptive PGD)

The idea behind CAPGD is relatively simple: instead of performing a single projection (as in PGD), it performs a second one to respect the domain constraints typical of tabular data (e.g., immutable features such as IDs and timestamps, integer features etc).

Moreover, unlike PGD, in CAPGD the step size is adaptive. It halves if the loss does not increase after an iteration.

In [9]:
def repair_constraints(x_adv, x_clean,
                       immutability_mask=None,
                       categorical_indices=None,
                       lower_bounds=None,
                       upper_bounds=None):
                       
    """
    1. Feature immutabili -> ripristina i valori originali
    2. Feature categoriche -> arrotonda all'intero più vicino
    3. Bounds per feature
    """

    x_adv = x_adv.clone()

    if immutability_mask is not None:
        x_adv[:, immutability_mask] = x_clean[:, immutability_mask]

    if categorical_indices is not None:
        for idx in categorical_indices:
            x_adv[:, idx] = torch.round(x_adv[:, idx])

    if lower_bounds is not None:
        x_adv = torch.max(torch.min(x_adv, upper_bounds), lower_bounds)

    return x_adv


def capgd_attack(model, x_clean, y_true,
                 epsilon=0.5, n_iter=10,
                 immutability_mask=None,
                 categorical_indices=None,
                 lower_bounds=None,
                 upper_bounds=None,
                 min_eta=1e-4,
                 device='cuda'):

    model.eval()
    x_clean = x_clean.to(device)
    y_true = y_true.to(device)
    N, D = x_clean.shape

    # Inizializzazione random
    delta = torch.zeros_like(x_clean).uniform_(-epsilon, epsilon)
    x_adv = x_clean + delta
    x_adv = repair_constraints(x_adv, x_clean, immutability_mask,
                               categorical_indices, lower_bounds, upper_bounds)

    eta = 2 * epsilon
    loss_history = []

    for k in range(n_iter):
        x_adv = x_adv.detach().requires_grad_(True)

        logits = model(x_adv)
        loss = F.cross_entropy(logits, y_true)
        loss_history.append(loss.item())

        grad = torch.autograd.grad(loss, x_adv)[0]

        with torch.no_grad():

            # Aggiornamento
            x_adv = x_adv + eta * grad.sign()

            # Proiezione sulla palla L2 di raggio epsilon
            delta = x_adv - x_clean
            delta_norm = delta.view(N, -1).norm(p=2, dim=1).view(N, 1)
            factor = torch.clamp(epsilon / (delta_norm + 1e-8), max=1.0)
            delta = delta * factor
            x_adv = x_clean + delta

            # Riparazione vincoli di dominio
            x_adv = repair_constraints(x_adv, x_clean, immutability_mask,
                                       categorical_indices, lower_bounds, upper_bounds)

        # adaptive step size
        if k > 0 and loss_history[-1] <= loss_history[-2]:
            eta = max(eta / 2, min_eta)

    return x_adv.detach()

### Constraint definition

The features in the Elliptic dataset are anonymized, so we don't know which of them are immutable, categorical etc. We will therefore adopt one simple constrain: the attacker must respect the observed range of each feature in the trainign set.

In [10]:
lower_bounds = X_train_t.min(dim=0).values.to(device)
upper_bounds = X_train_t.max(dim=0).values.to(device)

immutability_mask = None
categorical_indices = None

### Attack execution

In [11]:
X_test_dev = X_test_t.to(device)
y_test_dev = y_test_t.to(device)

x_adv = capgd_attack(
    model, X_test_dev, y_test_dev,
    epsilon=0.5,
    n_iter=20,
    immutability_mask=immutability_mask,
    categorical_indices=categorical_indices,
    lower_bounds=lower_bounds,
    upper_bounds=upper_bounds,
    device=device
)

### Evaluation

In [12]:
model.eval()
with torch.no_grad():
    clean_preds = model(X_test_dev).argmax(dim=1)
    adv_preds = model(x_adv).argmax(dim=1)

    clean_acc = (clean_preds == y_test_dev).float().mean().item()
    adv_acc = (adv_preds == y_test_dev).float().mean().item()

    originally_correct = (clean_preds == y_test_dev)
    attack_success = (adv_preds[originally_correct] != y_test_dev[originally_correct]).float().mean().item()

    # F1 sulle illicit
    clean_f1 = f1_score(y_test_t.numpy(), clean_preds.cpu().numpy(), pos_label=1)
    adv_f1 = f1_score(y_test_t.numpy(), adv_preds.cpu().numpy(), pos_label=1)

    # Norma L2 media della perturbazione
    l2_pert = (x_adv - X_test_dev).norm(p=2, dim=1).mean().item()

print(f"\nRisultati (epsilon={0.5}, n_iter=20)")
print(f"Clean accuracy: {clean_acc:.4f}")
print(f"Adversarial accuracy: {adv_acc:.4f}")
print(f"Attack success rate: {attack_success:.4f}")
print(f"Clean F1 (illicit): {clean_f1:.4f}")
print(f"Adversarial F1 (illicit): {adv_f1:.4f}")
print(f"Mean L2 perturbation: {l2_pert:.4f}")

print("\nClassification Report (adversarial)")
print(classification_report(y_test_t.numpy(), adv_preds.cpu().numpy(),
    target_names=['licit', 'illicit']))


Risultati (epsilon=0.5, n_iter=20)
Clean accuracy: 0.9575
Adversarial accuracy: 0.9352
Attack success rate: 0.0284
Clean F1 (illicit): 0.6395
Adversarial F1 (illicit): 0.5209
Mean L2 perturbation: 2.1665

Classification Report (adversarial)
              precision    recall  f1-score   support

       licit       0.97      0.96      0.97     15587
     illicit       0.50      0.54      0.52      1083

    accuracy                           0.94     16670
   macro avg       0.73      0.75      0.74     16670
weighted avg       0.94      0.94      0.94     16670

